# IMSE 111 · Week 1: IE by the numbers

**Throughput, bottlenecks, and takt time at the campus coffee cart**

This notebook goes with *Chapter 1: What Is Industrial & Manufacturing Engineering?*

**What you'll do:**
1. Compute the capacity of each step in a process.
2. Find the bottleneck and the system's maximum throughput.
3. Compare capacity with customer demand using takt time.
4. Test what happens when you add workers.

> Run each cell with **Shift + Enter**. Save your own copy first: **File → Save a copy in Drive**.

## 0. Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4)
NDSU_GREEN, NDSU_GOLD = "#0A5640", "#FFC82E"

## 1. Describe the process

Each step has a **time per customer** (seconds) and a number of **workers** at that step.

In [ ]:
process = pd.DataFrame({
    "step":    ["Take order", "Make drink", "Pay"],
    "seconds": [30, 75, 20],
    "workers": [1, 1, 1],
})
process

## 2. Capacity and the bottleneck

A step with time $t$ seconds and $w$ workers can serve at most $3600 \cdot w / t$ customers per hour.
The step with the **lowest** capacity is the **bottleneck**, and it sets the throughput of the whole system.

In [ ]:
def analyze(process, demand_per_hour):
    df = process.copy()
    df["capacity_per_hr"] = 3600 * df["workers"] / df["seconds"]
    throughput = df["capacity_per_hr"].min()
    served = min(throughput, demand_per_hour)
    df["utilization"] = served / df["capacity_per_hr"]
    bottleneck = df.loc[df["capacity_per_hr"].idxmin(), "step"]
    return df, throughput, bottleneck

demand = 60  # customers per hour during the morning rush
table, throughput, bottleneck = analyze(process, demand)
print(f"Bottleneck: {bottleneck}")
print(f"Max throughput: {throughput:.0f} customers/hour (demand = {demand})")
print(f"Takt time: {3600 / demand:.0f} s per customer")
table.round(2)

In [ ]:
colors = [NDSU_GOLD if s == bottleneck else NDSU_GREEN for s in table["step"]]
ax = table.plot.bar(x="step", y="capacity_per_hr", color=colors, legend=False)
ax.axhline(demand, color="crimson", linestyle="--", label=f"Demand = {demand}/hr")
ax.set_ylabel("Customers per hour")
ax.set_xlabel("")
ax.set_title("Capacity of each step (bottleneck in gold)")
ax.legend()
plt.xticks(rotation=0)
plt.show()

## 3. What if we add baristas?

Let's see how system throughput changes as we add workers to the *Make drink* step.

In [ ]:
rows = []
for baristas in range(1, 5):
    p = process.copy()
    p.loc[p["step"] == "Make drink", "workers"] = baristas
    _, tp, bn = analyze(p, demand)
    rows.append({"baristas": baristas, "throughput_per_hr": round(tp, 1), "bottleneck": bn})
pd.DataFrame(rows)

## 4. Try it yourself

1. Demand during finals week rises to **100 customers/hour**. How many baristas are needed, and which step becomes the new bottleneck?
2. A mobile-ordering app cuts *Take order* to **10 s**. Re-run the analysis. Does it help during the rush? Why or why not?

In [ ]:
# your answer here

## Reflection

In two or three sentences: why doesn't speeding up a non-bottleneck step increase the system's throughput?